In [1]:
import os

SRC_DIR = "/kaggle/input/datasets/swetha344/pyg-210-cu128-wheels"
DST_DIR = "/kaggle/working/fixed_wheels"

os.makedirs(DST_DIR, exist_ok=True)

rename_map = {
    "torch_scatter-2.1.2pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_cluster-1.6.3pt210cu128-cp312-cp312-linux_x86_64.whl": "torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "pyg_lib-0.4.0pt210cu128-cp312-cp312-linux_x86_64.whl": "pyg_lib-0.4.0+pt210cu128-cp312-cp312-linux_x86_64.whl",
    "torch_geometric-2.7.0-py3-none-any.whl": "torch_geometric-2.7.0-py3-none-any.whl",
    "ogb-1.3.6-py3-none-any.whl": "ogb-1.3.6-py3-none-any.whl"
}

for old_name, new_name in rename_map.items():
    src = os.path.join(SRC_DIR, old_name)
    dst = os.path.join(DST_DIR, new_name)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

!pip install --no-index --no-deps --force-reinstall /kaggle/working/fixed_wheels/*.whl

Processing ./fixed_wheels/ogb-1.3.6-py3-none-any.whl
Processing ./fixed_wheels/torch_cluster-1.6.3+pt210cu128-cp312-cp312-linux_x86_64.whl
Processing ./fixed_wheels/torch_geometric-2.7.0-py3-none-any.whl
Processing ./fixed_wheels/torch_scatter-2.1.2+pt210cu128-cp312-cp312-linux_x86_64.whl


In [2]:
import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

In [3]:
import torch
import torch_geometric

from torch_geometric.data.data import DataEdgeAttr
from torch_geometric.data.storage import NodeStorage, EdgeStorage, GlobalStorage

torch.serialization.add_safe_globals([
    DataEdgeAttr, 
    NodeStorage, 
    EdgeStorage, 
    GlobalStorage,
    torch_geometric.data.Data
])

from ogb.nodeproppred import PygNodePropPredDataset
import torch_geometric.transforms as T

with torch.serialization.safe_globals([DataEdgeAttr, NodeStorage, EdgeStorage]):
    dataset = PygNodePropPredDataset(name='ogbn-arxiv', root='dataset/', transform=T.ToUndirected())

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:02<00:00, 32.17it/s]
Processing...


Extracting dataset/arxiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 10433.59it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 318.52it/s]

Saving...



Done!
/usr/local/lib/python3.12/dist-packages/ogb/nodeproppred/dataset_pyg.py:69: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.data, self.slices = torch.load(self.processed_paths[0])


In [4]:
# 1. Basic Dataset Info
print(f"Number of graphs: {len(dataset)}")
print(f"Number of classes: {dataset.num_classes}")
print(f"Number of node features: {dataset.num_node_features}")

# 2. Get the actual Data object
data = dataset[0]

# 3. View Graph Structure
print("\n--- Graph Structure ---")
print(f"Number of nodes: {data.num_nodes}")
print(f"Number of edges: {data.num_edges}")
print(f"Is undirected: {data.is_undirected()}")
print(f"Has isolated nodes: {data.has_isolated_nodes()}")
print(f"Has self-loops: {data.has_self_loops()}")

# 4. Print one Node's Features
# Nodes are indexed. Let's look at node 0.
print("\n--- Node 0 Inspection ---")
print(f"Features (first 5 values): {data.x[0][:5]}")
print(f"Label: {data.y[0].item()}")

# 5. Inspect the Edge Index
# This is a [2, num_edges] tensor representing (source, target)
print("\n--- Edge Index ---")
print(data.edge_index)

Number of graphs: 1
Number of classes: 40
Number of node features: 128

--- Graph Structure ---
Number of nodes: 169343
Number of edges: 2315598
Is undirected: True
Has isolated nodes: False
Has self-loops: False

--- Node 0 Inspection ---
Features (first 5 values): tensor([-0.0579, -0.0525, -0.0726, -0.0266,  0.1304])
Label: 4

--- Edge Index ---
tensor([[     0,      0,      0,  ..., 169341, 169342, 169342],
        [   411,    640,   1162,  ..., 163274,  27824, 158981]])


In [5]:
split_idx = dataset.get_idx_split()
train_idx, valid_idx, test_idx = split_idx["train"], split_idx["valid"], split_idx["test"]

print(f"\nTraining nodes: {len(train_idx)}")
print(f"Validation nodes: {len(valid_idx)}")
print(f"Test nodes: {len(test_idx)}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)


Training nodes: 90941
Validation nodes: 29799
Test nodes: 48603


In [6]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import torch.nn as nn

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers, dropout):
        super(GCN, self).__init__()
        
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        
        # First layer
        self.convs.append(GCNConv(in_channels, hidden_channels))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        
        # Hidden layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Output layer
        self.convs.append(GCNConv(hidden_channels, out_channels))
        
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i in range(len(self.convs) - 1):
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
            
        x = self.convs[-1](x, edge_index)
        return x # Return logits (LogSoftmax usually handled by loss function)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = torch.nn.CrossEntropyLoss()

In [7]:
import torch
import numpy as np

# Configuration
num_runs = 10
epochs = 500
all_test_results = []

global_best_test_acc = 0.0
global_save_path = 'overall_best_gcn_model.pt'


for run in range(num_runs):
    # 1. Set a different seed for each run
    seed = run # 0, 1, 2... 9
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    
    print(f"\n>>> Starting Run {run+1}/{num_runs} (Seed: {seed})")
    
    # 2. Re-initialize the model and optimizer inside the loop
    model = GCN(
        in_channels=dataset.num_node_features, 
        hidden_channels=256, 
        out_channels=dataset.num_classes, 
        num_layers=3, 
        dropout=0.5
    ).to(device)

    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    best_run_val_acc = 0
    best_run_test_acc = 0

    # 3. Training Loop for the current run
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[train_idx], data.y[train_idx].squeeze())
        loss.backward()
        optimizer.step()

        # Check performance periodically
        if epoch % 50 == 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                y_pred = logits.argmax(dim=-1, keepdim=True)
                
                val_acc = (y_pred[valid_idx] == data.y[valid_idx]).sum().item() / valid_idx.size(0)
                test_acc = (y_pred[test_idx] == data.y[test_idx]).sum().item() / test_idx.size(0)
                
                # Keep track of the test accuracy corresponding to the best validation accuracy
                if val_acc > best_run_val_acc:
                    best_run_val_acc = val_acc
                    best_run_test_acc = test_acc

                    if test_acc > global_best_test_acc:
                        global_best_test_acc = test_acc
                        torch.save(model.state_dict(), global_save_path)

    all_test_results.append(best_run_test_acc)
    print(f"Run {run+1} Finished. Best Test Acc: {best_run_test_acc:.4f}")

# 4. Final Reporting
mean_acc = np.mean(all_test_results)
std_acc = np.std(all_test_results)

print("\n" + "="*30)
print(f"FINAL METRICS OVER {num_runs} RUNS")
print(f"Test Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print("="*30)


>>> Starting Run 1/10 (Seed: 0)
Run 1 Finished. Best Test Acc: 0.7187

>>> Starting Run 2/10 (Seed: 1)
Run 2 Finished. Best Test Acc: 0.7190

>>> Starting Run 3/10 (Seed: 2)
Run 3 Finished. Best Test Acc: 0.7104

>>> Starting Run 4/10 (Seed: 3)
Run 4 Finished. Best Test Acc: 0.7111

>>> Starting Run 5/10 (Seed: 4)
Run 5 Finished. Best Test Acc: 0.7223

>>> Starting Run 6/10 (Seed: 5)
Run 6 Finished. Best Test Acc: 0.7131

>>> Starting Run 7/10 (Seed: 6)
Run 7 Finished. Best Test Acc: 0.7172

>>> Starting Run 8/10 (Seed: 7)
Run 8 Finished. Best Test Acc: 0.7132

>>> Starting Run 9/10 (Seed: 8)
Run 9 Finished. Best Test Acc: 0.7156

>>> Starting Run 10/10 (Seed: 9)
Run 10 Finished. Best Test Acc: 0.7123

FINAL METRICS OVER 10 RUNS
Test Accuracy: 0.7153 ± 0.0037
